# Regenerate Multi-Metric Distance Files

This notebook regenerates `multi_metric_distances.parquet` files for all datasets using the unified pipeline.

**Why?** The old `analyze_simulation_multi_metric()` function is deprecated. The new unified pipeline (`compare-results --multi-metric`) provides the same functionality with these improvements:
- Includes `all_tools_combined` synthetic tool
- Adds `tools` column (list of tools that found each alignment)
- Consistent handling of invalid alignments across metrics
- Same output format for notebook compatibility

## Output Files
For each dataset directory, this creates:
- `multi_metric_distances.parquet` - Region-level data with all distance metrics and tools
- `performance_results_{metric}_mm5.tsv` - Per-tool performance summary
- `tools_results.tsv` - All tool alignments with classifications

In [2]:
import subprocess
from pathlib import Path
import polars as pl
from IPython.display import display

## Configuration

In [3]:
# Project paths
PROJECT_DIR = Path.cwd().parent
SIMULATED_DIR = PROJECT_DIR / "results" / "simulated"
REAL_DATA_DIR = PROJECT_DIR / "results" / "real_data" / "subsamples"
IMGVR4_SPACERS = PROJECT_DIR / "imgvr4_data" / "spacers" / "iphop_filtered_spacers.fna"

# Parameters
MAX_MISMATCHES = 5
SKIP_HYPERFINE = True  # Skip benchmarking for faster runs
VERBOSE = False  # Set to True for detailed logs

print(f"Project directory: {PROJECT_DIR}")
print(f"Simulated data: {SIMULATED_DIR}")
print(f"Real data: {REAL_DATA_DIR}")

Project directory: /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench
Simulated data: /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/simulated
Real data: /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/real_data/subsamples


## Helper Function

In [4]:
def run_compare_results(input_dir, contigs=None, spacers=None, distance_metric="hamming"):
    """
    Run compare-results with multi-metric mode.
    
    Args:
        input_dir: Path to dataset directory
        contigs: Optional path to contigs file (for real data)
        spacers: Optional path to spacers file (for real data)
        distance_metric: Default metric (hamming or edit)
    """
    # Create logs directory
    logs_dir = Path(input_dir) / "logs"
    logs_dir.mkdir(exist_ok=True)
    
    logfile = logs_dir / f"compare_results_mm{MAX_MISMATCHES}.log"
    
    cmd = [
        "pixi", "run", "spacer_bencher", "compare-results",
        "-i", str(input_dir),
        "-mm", str(MAX_MISMATCHES),
        "--multi-metric",
        "--distance", distance_metric,
        "--logfile", str(logfile),
    ]
    
    if SKIP_HYPERFINE:
        cmd.append("--skip-hyperfine")
    
    if VERBOSE:
        cmd.append("--verbose")
    
    if contigs:
        cmd.extend(["--contigs", str(contigs)])
    
    if spacers:
        cmd.extend(["--spacers", str(spacers)])
    
    print(f"  Running: {' '.join(cmd[2:])}")
    print(f"  Logfile: {logfile.relative_to(PROJECT_DIR)}")
    
    result = subprocess.run(
        cmd,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        print(f"  ✗ ERROR: {result.stderr}")
        print(f"  See logfile for details: {logfile}")
        return False
    else:
        # Extract key output lines
        for line in result.stdout.split('\n'):
            if 'Wrote' in line or 'multi_metric' in line or 'INFO' in line:
                print(f"  {line.strip()}")
        print(f"  ✓ Log written to: {logfile.relative_to(PROJECT_DIR)}")
        return True

## Process Simulated Datasets

In [9]:
print("PROCESSING SIMULATED DATASETS")

simulated_dirs = sorted([d for d in SIMULATED_DIR.iterdir() if d.is_dir()])
print(f"Found {len(simulated_dirs)} simulation directories\n")

processed = []
skipped = []
failed = []

for sim_dir in simulated_dirs:
    sim_name = sim_dir.name
    
    # Check requirements
    if not (sim_dir / "simulated_data").exists():
        print(f"⊘ Skipping {sim_name}: no simulated_data/")
        skipped.append(sim_name)
        continue
    
    if not (sim_dir / "raw_outputs").exists():
        print(f"⊘ Skipping {sim_name}: no raw_outputs/")
        skipped.append(sim_name)
        continue
    
    print(f"→ Processing: {sim_name}")
    
    success = run_compare_results(
        input_dir=sim_dir,
        distance_metric="hamming"  # Default 
    )
    
    if success:
        print(f"✓ Completed: {sim_name}\n")
        processed.append(sim_name)
    else:
        failed.append(sim_name)
    
print("\n" + "="*60)
print(f"Processed: {len(processed)}")
print(f"Skipped: {len(skipped)}")
print(f"Failed: {len(failed)}")
if failed:
    print(f"Failed datasets: {failed}")

PROCESSING SIMULATED DATASETS
Found 9 simulation directories

→ Processing: ns_100000_nc_10000
  Running: spacer_bencher compare-results -i /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/simulated/ns_100000_nc_10000 -mm 5 --multi-metric --distance hamming --logfile /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/simulated/ns_100000_nc_10000/logs/compare_results_mm5.log --skip-hyperfine
  Logfile: results/simulated/ns_100000_nc_10000/logs/compare_results_mm5.log
  [02/12/26 21:41:56] INFO     DEBUG logging enabled to file: /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/simulated/ns_100000_nc_10000/logs/compare_results_mm5.log
  [02/12/26 21:41:56] INFO     Comparing tool results in /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/simulated/ns_100000_nc_10000
  [02/12/26 21:41:56] INFO     Multi-metric mode enabled: calculating performance for both hamming and edit distance
  

## Process Real Data Subsamples

In [ ]:
print("PROCESSING REAL DATA SUBSAMPLES")

if not REAL_DATA_DIR.exists():
    print(f"⊘ Real data directory not found: {REAL_DATA_DIR}")
else:
    fraction_dirs = sorted([d for d in REAL_DATA_DIR.iterdir() if d.is_dir() and d.name.startswith('fraction_')])
    print(f"Found {len(fraction_dirs)} fraction directories\n")
    
    processed_real = []
    skipped_real = []
    failed_real = []
    
    for fraction_dir in fraction_dirs:
        fraction_name = fraction_dir.name
        
        # Check requirements
        if not (fraction_dir / "subsampled_data").exists():
            print(f"⊘ Skipping {fraction_name}: no subsampled_data/")
            skipped_real.append(fraction_name)
            continue
        
        if not (fraction_dir / "raw_outputs").exists():
            print(f"⊘ Skipping {fraction_name}: no raw_outputs/")
            skipped_real.append(fraction_name)
            continue
        
        contigs_file = fraction_dir / "subsampled_data" / "subsampled_contigs.fa"
        if not contigs_file.exists():
            print(f"⊘ Skipping {fraction_name}: no {contigs_file}")
            skipped_real.append(fraction_name)
            continue
        
        print(f"→ Processing: {fraction_name}")
        
        success = run_compare_results(
            input_dir=fraction_dir,
            contigs=contigs_file,
            spacers=IMGVR4_SPACERS,
            distance_metric="hamming"  # Real data uses hamming distance 
        )
        
        if success:
            print(f"✓ Completed: {fraction_name}\n")
            processed_real.append(fraction_name)
        else:
            failed_real.append(fraction_name)
    
    print(f"Processed: {len(processed_real)}")
    print(f"Skipped: {len(skipped_real)}")
    print(f"Failed: {len(failed_real)}")
    if failed_real:
        print(f"Failed fractions: {failed_real}")

PROCESSING REAL DATA SUBSAMPLES
Found 7 fraction directories

→ Processing: fraction_0.0005
  Running: spacer_bencher compare-results -i /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/real_data/subsamples/fraction_0.0005 -mm 5 --multi-metric --distance hamming --logfile /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/real_data/subsamples/fraction_0.0005/logs/compare_results_mm5.log --skip-hyperfine --contigs /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/real_data/subsamples/fraction_0.0005/subsampled_data/subsampled_contigs.fa --spacers /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/imgvr4_data/spacers/iphop_filtered_spacers.fna
  Logfile: results/real_data/subsamples/fraction_0.0005/logs/compare_results_mm5.log
  [02/12/26 21:57:31] INFO     DEBUG logging enabled to file: /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/real_data/subsamples/fraction_0.

In [ ]:
fraction_dirs = sorted([d for d in REAL_DATA_DIR.iterdir() if d.is_dir() and d.name.startswith('fraction_1')])
print(f"Found {len(fraction_dirs)} fraction directories\n")

processed_real = []
skipped_real = []
failed_real = []


fraction_dir = fraction_dirs[0]
fraction_name = fraction_dir.name

# Check requirements
if not (fraction_dir / "subsampled_data").exists():
    print(f"⊘ Skipping {fraction_name}: no subsampled_data/")
    skipped_real.append(fraction_name)

if not (fraction_dir / "raw_outputs").exists():
    print(f"⊘ Skipping {fraction_name}: no raw_outputs/")
    skipped_real.append(fraction_name)

contigs_file = fraction_dir / "subsampled_data" / "subsampled_contigs.fa"
if not contigs_file.exists():
    print(f"⊘ Skipping {fraction_name}: no {contigs_file}")
    skipped_real.append(fraction_name)

print(f"→ Processing: {fraction_name}")

success = run_compare_results(
    input_dir=fraction_dir,
    contigs=contigs_file,
    spacers=IMGVR4_SPACERS,
    distance_metric="hamming"  # Real data uses hamming distance 
)

if success:
    print(f"✓ Completed: {fraction_name}\n")
    processed_real.append(fraction_name)
else:
    failed_real.append(fraction_name)

Found 1 fraction directories

→ Processing: fraction_1
  Running: spacer_bencher compare-results -i /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/real_data/subsamples/fraction_1 -mm 5 --multi-metric --distance hamming --logfile /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/real_data/subsamples/fraction_1/logs/compare_results_mm5.log --skip-hyperfine --contigs /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/results/real_data/subsamples/fraction_1/subsampled_data/subsampled_contigs.fa --spacers /clusterfs/jgi/scratch/science/metagen/neri/code/blits/spacer_bench/imgvr4_data/spacers/iphop_filtered_spacers.fna
  Logfile: results/real_data/subsamples/fraction_1/logs/compare_results_mm5.log


## Verify Output Files

Check that the output files have the expected structure for notebook compatibility.

In [5]:
# Pick a sample simulation to verify
sample_sim = None
for sim_dir in SIMULATED_DIR.iterdir():
    if (sim_dir / "multi_metric_distances.parquet").exists():
        sample_sim = sim_dir
        break

if sample_sim:
    print(f"Verifying output for: {sample_sim.name}\n")
    
    # Load multi_metric file
    parquet_file = sample_sim / "multi_metric_distances.parquet"
    df = pl.read_parquet(parquet_file)
    
    print(f"File: {parquet_file.relative_to(PROJECT_DIR)}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {df.columns}")
    
    # Check for required columns
    required_cols = ['alignment_idx', 'region_idx', 'classification', 'tools', 
                     'distance_hamming', 'distance_edit', 'distance_gap_affine']
    missing = [col for col in required_cols if col not in df.columns]
    
    if missing:
        print(f"\n⚠ WARNING: Missing columns: {missing}")
    else:
        print(f"\n✓ All required columns present")
    
    # Check tools column
    if 'tools' in df.columns:
        print(f"\n'tools' column type: {df['tools'].dtype}")
        
        # Sample tools values
        print("\nSample 'tools' values:")
        display(df.select(['alignment_idx', 'classification', 'tools']).head(5))
        
        # Check if all_tools_combined is present
        tools_exploded = df.explode('tools')
        unique_tools = sorted(tools_exploded['tools'].unique().to_list())
        print(f"\nUnique tools found: {unique_tools}")
        
        if 'all_tools_combined' in unique_tools:
            print("✓ all_tools_combined is present")
        else:
            print("✗ ERROR: all_tools_combined is missing!")
    
    # Check performance file
    perf_file = sample_sim / "performance_results_hamming_mm5.tsv"
    if perf_file.exists():
        print(f"\n\nPerformance file: {perf_file.relative_to(PROJECT_DIR)}")
        perf = pl.read_csv(perf_file, separator='\t')
        print(f"Tools in performance results: {sorted(perf['tool'].to_list())}")
        
        if 'all_tools_combined' in perf['tool'].to_list():
            print("✓ all_tools_combined in performance results")
            combined_perf = perf.filter(pl.col('tool') == 'all_tools_combined')
            print(f"\nall_tools_combined metrics:")
            display(combined_perf)
        else:
            print("✗ ERROR: all_tools_combined not in performance results!")
else:
    print("No simulation with multi_metric_distances.parquet found - run cells above first")

Verifying output for: ns_100000_nc_10000

File: results/simulated/ns_100000_nc_10000/multi_metric_distances.parquet
Shape: (890226, 17)

Columns: ['alignment_idx', 'region_idx', 'spacer_id', 'contig_id', 'start', 'end', 'strand', 'classification', 'start_gt', 'end_gt', 'mismatches_gt', 'planned_mismatches', 'distance_hamming', 'distance_edit', 'distance_gap_affine', 'recalculated_distance', 'tools']

✓ All required columns present

'tools' column type: List(String)

Sample 'tools' values:


alignment_idx,classification,tools
u64,str,list[str]
62776,"""positive_in_plan""","[""all_tools_combined"", ""blastn"", … ""x_mapper""]"
78876,"""positive_not_in_plan""","[""all_tools_combined"", ""sassy""]"
67819,"""positive_not_in_plan""","[""all_tools_combined"", ""sassy""]"
68711,"""positive_not_in_plan""","[""all_tools_combined"", ""sassy""]"
56832,"""positive_in_plan""","[""all_tools_combined"", ""indelfree_bruteforce"", … ""sassy""]"



Unique tools found: ['all_tools_combined', 'blastn', 'bowtie1', 'bowtie2', 'indelfree_bruteforce', 'indelfree_indexed', 'minimap2', 'mmseqs2', 'mummer4', 'sassy', 'strobealign', 'x_mapper']
✓ all_tools_combined is present


Performance file: results/simulated/ns_100000_nc_10000/performance_results_hamming_mm5.tsv
Tools in performance results: ['all_tools_combined', 'blastn', 'bowtie1', 'bowtie2', 'indelfree_bruteforce', 'indelfree_indexed', 'minimap2', 'mmseqs2', 'mummer4', 'sassy', 'strobealign', 'x_mapper']
✓ all_tools_combined in performance results

all_tools_combined metrics:


tool,planned_true_positives_hamming,invalid_alignments_hamming,positives_not_in_plan_hamming,ground_truth_planned,ground_truth_augmented_hamming,all_true_positives_hamming,false_negatives_planned_hamming,false_negatives_augmented_hamming,recall_planned_hamming,recall_augmented_hamming,tool_right,planned_true_positives_edit,invalid_alignments_edit,positives_not_in_plan_edit,ground_truth_planned_right,ground_truth_augmented_edit,all_true_positives_edit,false_negatives_planned_edit,false_negatives_augmented_edit,recall_planned_edit,recall_augmented_edit
str,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,str,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64
"""all_tools_combined""",299499,477903,21776,299499,321275,321275,0,0,1.0,1.0,"""all_tools_combined""",299499,14,475292,299499,774791,774791,0,0,1.0,1.0


## Summary

All datasets now have:
- ✓ `multi_metric_distances.parquet` with `tools` column and `all_tools_combined`
- ✓ `performance_results_{metric}_mm5.tsv` with per-tool metrics
- ✓ `tools_results.tsv` with all alignments

**Notebooks can now use these files directly** without needing `analyze_simulation_multi_metric()`.